In [1]:
import rasterio
import numpy as np
from pathlib import Path

tif_path = Path("/mnt/disk1/workspace_jym/LCZ/data/GT/seoul_LCZ.tif")

print("File exists:", tif_path.exists())
print("Path:", tif_path)

with rasterio.open(tif_path) as src:
    print("\n========== Basic Info ==========")
    print("Driver:", src.driver)
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Band count:", src.count)
    print("Dtype:", src.dtypes)
    print("Nodata:", src.nodata)

    print("\n========== Spatial Info ==========")
    print("Transform:", src.transform)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Pixel size X:", src.transform.a)
    print("Pixel size Y:", src.transform.e)

    arr = src.read(1)

print("\n========== Array Info ==========")
print("Shape:", arr.shape)
print("Min:", np.nanmin(arr))
print("Max:", np.nanmax(arr))

nodata = src.nodata
if nodata is not None:
    valid = arr[arr != nodata]
else:
    valid = arr

unique_vals, counts = np.unique(valid, return_counts=True)

print("\n========== Unique Values ==========")
print("Number of unique values:", len(unique_vals))
print("Unique values:")
print(unique_vals)

print("\n========== Value Counts ==========")
for v, c in zip(unique_vals, counts):
    print(f"Value {v}: {c} pixels")

File exists: True
Path: /mnt/disk1/workspace_jym/LCZ/data/GT/seoul_LCZ.tif

========== Basic Info ==========
Driver: GTiff
CRS: EPSG:32652
Width: 899
Height: 849
Band count: 1
Dtype: ('uint32',)
Nodata: 2147483647.0

========== Spatial Info ==========
Transform: | 50.00, 0.00, 300630.00|
| 0.00,-50.00, 4180100.00|
| 0.00, 0.00, 1.00|
Resolution: (50.0, 50.0)
Bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)
Pixel size X: 50.0
Pixel size Y: -50.0

========== Array Info ==========
Shape: (849, 899)
Min: 1
Max: 2147483647

========== Unique Values ==========
Number of unique values: 11
Unique values:
[  1   2   3   4   5   6   8 101 102 104 107]

========== Value Counts ==========
Value 1: 322 pixels
Value 2: 2052 pixels
Value 3: 6219 pixels
Value 4: 5155 pixels
Value 5: 815 pixels
Value 6: 925 pixels
Value 8: 899 pixels
Value 101: 13410 pixels
Value 102: 4565 pixels
Value 104: 4637 pixels
Value 107: 2547 pixels


In [2]:
import rasterio
import numpy as np
from scipy.ndimage import label
from pathlib import Path

tif_path = Path("/mnt/disk1/workspace_jym/LCZ/data/GT/seoul_LCZ.tif")

with rasterio.open(tif_path) as src:
    arr = src.read(1)
    nodata = src.nodata

if nodata is not None:
    valid_mask = arr != nodata
else:
    valid_mask = np.ones(arr.shape, dtype=bool)

unique_vals = np.unique(arr[valid_mask])

print("========== LCZ Class / Connected Component Check ==========")

total_components = 0

for cls in unique_vals:
    class_mask = (arr == cls)
    
    # 8-neighborhood connected components
    structure = np.ones((3, 3), dtype=np.int8)
    labeled, num_components = label(class_mask, structure=structure)
    
    pixel_count = class_mask.sum()
    total_components += num_components
    
    print(f"Class value {cls}: pixels={pixel_count}, connected_components={num_components}")

print("\nTotal connected components over all class values:", total_components)
print("Number of unique raster values:", len(unique_vals))

========== LCZ Class / Connected Component Check ==========
Class value 1: pixels=322, connected_components=7
Class value 2: pixels=2052, connected_components=55
Class value 3: pixels=6219, connected_components=112
Class value 4: pixels=5155, connected_components=95
Class value 5: pixels=815, connected_components=22
Class value 6: pixels=925, connected_components=14
Class value 8: pixels=899, connected_components=10
Class value 101: pixels=13410, connected_components=38
Class value 102: pixels=4565, connected_components=55
Class value 104: pixels=4637, connected_components=20
Class value 107: pixels=2547, connected_components=19

Total connected components over all class values: 447
Number of unique raster values: 11


In [3]:
import rasterio
import numpy as np
import pandas as pd
from scipy.ndimage import label
from pathlib import Path

gt_path = Path("/mnt/disk1/workspace_jym/LCZ/data/GT/seoul_LCZ.tif")
out_poly_path = Path("/mnt/disk1/workspace_jym/LCZ/data/GT/lcz_polygonnumber_50m.tif")
out_table_path = Path("/mnt/disk1/workspace_jym/LCZ/data/GT/LCZ_class_from_components.csv")

with rasterio.open(gt_path) as src:
    lcz = src.read(1)
    profile = src.profile.copy()
    nodata = src.nodata

print("Nodata:", nodata)
print("Input shape:", lcz.shape)

# nodata 제외
valid_mask = lcz != nodata

# 실제 LCZ class 값만 추출
classes = np.unique(lcz[valid_mask])
classes = classes[classes > 0]

print("LCZ classes:", classes)

# polygon number raster
poly_num = np.zeros(lcz.shape, dtype=np.uint32)

# 8-neighborhood 연결성 사용
structure = np.ones((3, 3), dtype=np.int8)

records = []
current_id = 1

for cls in classes:
    class_mask = (lcz == cls)
    labeled, num_components = label(class_mask, structure=structure)

    print(f"Class {cls}: {num_components} components")

    for comp_id in range(1, num_components + 1):
        comp_mask = labeled == comp_id
        pixel_count = int(comp_mask.sum())

        poly_num[comp_mask] = current_id

        records.append({
            "polygon_id": current_id,
            "pixel_count": pixel_count,
            "lcz_class": int(cls)
        })

        current_id += 1

print("Total polygon IDs:", current_id - 1)

# 저장 profile 설정
profile.update(
    dtype=rasterio.uint32,
    count=1,
    nodata=0,
    compress="lzw"
)

with rasterio.open(out_poly_path, "w", **profile) as dst:
    dst.write(poly_num, 1)

df = pd.DataFrame(records)
df.to_csv(out_table_path, index=False, encoding="utf-8-sig")

print("Saved polygon number raster:", out_poly_path)
print("Saved polygon-class table:", out_table_path)

# 검증
with rasterio.open(out_poly_path) as src:
    check = src.read(1)
    print("\n========== Output Check ==========")
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Shape:", check.shape)
    print("Nodata:", src.nodata)
    print("Min:", check.min())
    print("Max:", check.max())
    print("Unique polygon IDs excluding 0:", len(np.unique(check[check > 0])))

Nodata: 2147483647.0
Input shape: (849, 899)
LCZ classes: [  1   2   3   4   5   6   8 101 102 104 107]
Class 1: 7 components
Class 2: 55 components
Class 3: 112 components
Class 4: 95 components
Class 5: 22 components
Class 6: 14 components
Class 8: 10 components
Class 101: 38 components
Class 102: 55 components
Class 104: 20 components
Class 107: 19 components
Total polygon IDs: 447
Saved polygon number raster: /mnt/disk1/workspace_jym/LCZ/data/GT/lcz_polygonnumber_50m.tif
Saved polygon-class table: /mnt/disk1/workspace_jym/LCZ/data/GT/LCZ_class_from_components.csv

========== Output Check ==========
CRS: EPSG:32652
Resolution: (50.0, 50.0)
Bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)
Shape: (849, 899)
Nodata: 0.0
Min: 0
Max: 447
Unique polygon IDs excluding 0: 447


### Class Excel 만들기

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("/mnt/disk1/workspace_jym/LCZ/data/GT/LCZ_class_from_components.csv")

out_dir = Path("/mnt/disk1/workspace_jym/LCZ/data/class_excel")
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)

# 기존 LCZ_class.xls와 유사한 형태 저장
# col1: polygon_id, col2: pixel_count, col3: lcz_class
lcz_class_xlsx = out_dir / "LCZ_class.xlsx"
df[["polygon_id", "pixel_count", "lcz_class"]].to_excel(
    lcz_class_xlsx,
    index=False,
    header=False
)

rng = np.random.default_rng(42)

# 현재 서울 GT에 실제 존재하는 class
classes = sorted(df["lcz_class"].unique())

print("Classes:", classes)

for cls in classes:
    cls_df = df[df["lcz_class"] == cls].copy()
    cls_df = cls_df.sample(frac=1, random_state=42).reset_index(drop=True)

    n_poly = len(cls_df)
    split = int(np.ceil(n_poly * 0.5))

    train_df = cls_df.iloc[:split].copy()
    test_df = cls_df.iloc[split:].copy()

    # 기존 step1-2 코드가 첫 번째 열만 polygon_id로 읽기 때문에 header 없이 저장
    train_df[["polygon_id"]].to_excel(
        out_dir / f"{cls}_train.xlsx",
        index=False,
        header=False
    )

    test_df[["polygon_id"]].to_excel(
        out_dir / f"{cls}_test.xlsx",
        index=False,
        header=False
    )

    # red class용 all 파일도 만들어둠
    cls_df[["polygon_id"]].to_excel(
        out_dir / f"{cls}_all.xlsx",
        index=False,
        header=False
    )

    print(
        f"Class {cls}: total={n_poly}, "
        f"train={len(train_df)}, test={len(test_df)}"
    )

print("Saved to:", out_dir)

Classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(101), np.int64(102), np.int64(104), np.int64(107)]
Class 1: total=7, train=4, test=3
Class 2: total=55, train=28, test=27
Class 3: total=112, train=56, test=56
Class 4: total=95, train=48, test=47
Class 5: total=22, train=11, test=11
Class 6: total=14, train=7, test=7
Class 8: total=10, train=5, test=5
Class 101: total=38, train=19, test=19
Class 102: total=55, train=28, test=27
Class 104: total=20, train=10, test=10
Class 107: total=19, train=10, test=9
Saved to: /mnt/disk1/workspace_jym/LCZ/data/class_excel


### step1-2_LCZ_cali_vali.m

In [6]:
import numpy as np
import pandas as pd
import rasterio
from scipy.io import savemat
from pathlib import Path

# ============================================================
# step1-2_LCZ_cali_vali.py
# Python version for Seoul LCZ 50m GT
#
# Input:
#   - /mnt/disk1/workspace_jym/LCZ/data/GT/lcz_polygonnumber_50m.tif
#   - /mnt/disk1/workspace_jym/LCZ/data/GT/LCZ_class_from_components.csv
#
# Output:
#   - /mnt/disk1/workspace_jym/LCZ/data/mat_index/*_cali.mat
#   - /mnt/disk1/workspace_jym/LCZ/data/mat_index/*_vali.mat
#   - /mnt/disk1/workspace_jym/LCZ/data/mat_index/*_test.mat
# ============================================================

BASE_DIR = Path("/mnt/disk1/workspace_jym/LCZ/data")

GT_DIR = BASE_DIR / "GT"
OUT_DIR = BASE_DIR / "mat_index"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POLY_TIF = GT_DIR / "lcz_polygonnumber_50m.tif"
COMP_CSV = GT_DIR / "LCZ_class_from_components.csv"

# 기존 MATLAB 코드 기준:
# 일반 class: train polygon만 사용, pixel을 90% cali / 10% vali
# red class: all polygon 사용, pixel을 45% cali / 5% vali / 50% test
GENERAL_CLASSES = [1, 2, 5, 6, 8, 101, 102, 104, 107]
RED_CLASSES = [3, 4]

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


def matlab_linear_indices(mask: np.ndarray) -> np.ndarray:
    """
    Convert a boolean mask to MATLAB-style linear indices.

    MATLAB find() returns:
        1-based, column-major linear indices.

    For an array with shape (height, width):
        matlab_idx = row + col * height + 1
    """
    rows, cols = np.where(mask)
    height = mask.shape[0]
    idx = rows + cols * height + 1
    return idx.astype(np.int64).reshape(-1, 1)


def save_mat_vector(path: Path, var_name: str, values: np.ndarray):
    """
    Save a column vector to .mat file.
    """
    values = np.asarray(values, dtype=np.int64).reshape(-1, 1)
    savemat(path, {var_name: values})


# ============================================================
# 1. Load data
# ============================================================

if not POLY_TIF.exists():
    raise FileNotFoundError(f"Polygon number raster not found: {POLY_TIF}")

if not COMP_CSV.exists():
    raise FileNotFoundError(f"Component table not found: {COMP_CSV}")

with rasterio.open(POLY_TIF) as src:
    poly = src.read(1)
    print("========== Polygon Raster Info ==========")
    print("Path:", POLY_TIF)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Shape:", poly.shape)
    print("Nodata:", src.nodata)
    print("Min:", poly.min())
    print("Max:", poly.max())
    print("Unique polygon IDs excluding 0:", len(np.unique(poly[poly > 0])))

df = pd.read_csv(COMP_CSV)

required_cols = {"polygon_id", "pixel_count", "lcz_class"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing columns in CSV: {missing_cols}")

df["polygon_id"] = df["polygon_id"].astype(int)
df["lcz_class"] = df["lcz_class"].astype(int)
df["pixel_count"] = df["pixel_count"].astype(int)

print("\n========== Component CSV Info ==========")
print("Path:", COMP_CSV)
print("Number of polygons:", len(df))
print("Classes:", sorted(df["lcz_class"].unique().tolist()))

# ============================================================
# 2. Optional: save polygon split table for checking
# ============================================================

split_records = []

# ============================================================
# 3. General classes
#    - class별 polygon 중 50%를 train polygon으로 선택
#    - 각 train polygon 내부 pixel을 90% cali / 10% vali로 분할
# ============================================================

print("\n========== General Classes ==========")

for class_id in GENERAL_CLASSES:
    cls_df = df[df["lcz_class"] == class_id].copy()

    if cls_df.empty:
        print(f"[WARN] Class {class_id}: no polygons. Skip.")
        continue

    # polygon 순서 shuffle 후 50%를 train polygon으로 사용
    cls_df = cls_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    n_poly = len(cls_df)
    n_train_poly = int(np.ceil(n_poly * 0.5))

    train_df = cls_df.iloc[:n_train_poly].copy()

    print(f"\nClass {class_id}: total polygons={n_poly}, train polygons={len(train_df)}")

    for j, row in enumerate(train_df.itertuples(index=False), start=1):
        polygon_id = int(row.polygon_id)

        mask = poly == polygon_id
        want2 = matlab_linear_indices(mask)

        n_pix = want2.shape[0]
        if n_pix == 0:
            print(f"  [WARN] class={class_id}, polygon_id={polygon_id}: no pixels. Skip.")
            continue

        # MATLAB randperm에 해당
        perm = rng.permutation(n_pix)
        shf = want2[perm]

        cali_end = int(np.floor(n_pix * 0.9))

        # 너무 작은 polygon 예외 처리
        if cali_end < 1:
            cali = shf
            vali = np.empty((0, 1), dtype=np.int64)
        else:
            cali = shf[:cali_end]
            vali = shf[cali_end:]

        cali_path = OUT_DIR / f"{class_id}_{j}_cali.mat"
        vali_path = OUT_DIR / f"{class_id}_{j}_vali.mat"

        save_mat_vector(cali_path, "cali", cali)
        save_mat_vector(vali_path, "vali", vali)

        split_records.append({
            "lcz_class": class_id,
            "polygon_order": j,
            "polygon_id": polygon_id,
            "mode": "general",
            "total_pixels": n_pix,
            "cali_pixels": len(cali),
            "vali_pixels": len(vali),
            "test_pixels": 0,
            "cali_file": cali_path.name,
            "vali_file": vali_path.name,
            "test_file": ""
        })

        print(
            f"  polygon_order={j:03d}, polygon_id={polygon_id:04d}, "
            f"pixels={n_pix}, cali={len(cali)}, vali={len(vali)}"
        )

# ============================================================
# 4. Red / dense urban classes
#    - class별 모든 polygon 사용
#    - 각 polygon 내부 pixel을 45% cali / 5% vali / 50% test로 분할
# ============================================================

print("\n========== Red / Dense Urban Classes ==========")

for class_id in RED_CLASSES:
    cls_df = df[df["lcz_class"] == class_id].copy()

    if cls_df.empty:
        print(f"[WARN] Class {class_id}: no polygons. Skip.")
        continue

    cls_df = cls_df.sort_values("polygon_id").reset_index(drop=True)

    print(f"\nClass {class_id}: all polygons={len(cls_df)}")

    for j, row in enumerate(cls_df.itertuples(index=False), start=1):
        polygon_id = int(row.polygon_id)

        mask = poly == polygon_id
        want2 = matlab_linear_indices(mask)

        n_pix = want2.shape[0]
        if n_pix == 0:
            print(f"  [WARN] class={class_id}, polygon_id={polygon_id}: no pixels. Skip.")
            continue

        perm = rng.permutation(n_pix)
        shf = want2[perm]

        # 원본 MATLAB 코드와 동일한 비율
        cali_end = int(np.floor(n_pix * 0.45))
        test_start = int(np.floor(n_pix * 0.45) + np.ceil(n_pix * 0.05))

        # 너무 작은 polygon 예외 처리
        if n_pix < 3:
            cali = shf
            vali = np.empty((0, 1), dtype=np.int64)
            test = np.empty((0, 1), dtype=np.int64)
        else:
            cali_end = max(cali_end, 1)
            test_start = max(test_start, cali_end + 1)
            test_start = min(test_start, n_pix)

            cali = shf[:cali_end]
            vali = shf[cali_end:test_start]
            test = shf[test_start:]

        cali_path = OUT_DIR / f"{class_id}_{j}_cali.mat"
        vali_path = OUT_DIR / f"{class_id}_{j}_vali.mat"
        test_path = OUT_DIR / f"{class_id}_{j}_test.mat"

        save_mat_vector(cali_path, "cali", cali)
        save_mat_vector(vali_path, "vali", vali)
        save_mat_vector(test_path, "test", test)

        split_records.append({
            "lcz_class": class_id,
            "polygon_order": j,
            "polygon_id": polygon_id,
            "mode": "red",
            "total_pixels": n_pix,
            "cali_pixels": len(cali),
            "vali_pixels": len(vali),
            "test_pixels": len(test),
            "cali_file": cali_path.name,
            "vali_file": vali_path.name,
            "test_file": test_path.name
        })

        print(
            f"  polygon_order={j:03d}, polygon_id={polygon_id:04d}, "
            f"pixels={n_pix}, cali={len(cali)}, vali={len(vali)}, test={len(test)}"
        )

# ============================================================
# 5. Save summary
# ============================================================

summary_df = pd.DataFrame(split_records)
summary_path = OUT_DIR / "split_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\n========== Done ==========")
print("MAT files saved to:", OUT_DIR)
print("Summary CSV saved to:", summary_path)
print("Number of saved split records:", len(summary_df))

if not summary_df.empty:
    print("\nTotal pixels by split:")
    print("cali:", int(summary_df["cali_pixels"].sum()))
    print("vali:", int(summary_df["vali_pixels"].sum()))
    print("test:", int(summary_df["test_pixels"].sum()))

========== Polygon Raster Info ==========
Path: /mnt/disk1/workspace_jym/LCZ/data/GT/lcz_polygonnumber_50m.tif
CRS: EPSG:32652
Resolution: (50.0, 50.0)
Bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)
Shape: (849, 899)
Nodata: 0.0
Min: 0
Max: 447
Unique polygon IDs excluding 0: 447

========== Component CSV Info ==========
Path: /mnt/disk1/workspace_jym/LCZ/data/GT/LCZ_class_from_components.csv
Number of polygons: 447
Classes: [1, 2, 3, 4, 5, 6, 8, 101, 102, 104, 107]

========== General Classes ==========

Class 1: total polygons=7, train polygons=4
  polygon_order=001, polygon_id=0001, pixels=55, cali=49, vali=6
  polygon_order=002, polygon_id=0002, pixels=33, cali=29, vali=4
  polygon_order=003, polygon_id=0006, pixels=19, cali=17, vali=2
  polygon_order=004, polygon_id=0003, pixels=95, cali=85, vali=10

Class 2: total polygons=55, train polygons=28
  polygon_order=001, polygon_id=0039, pixels=73, cali=65, vali=8
  polygon_order=002, polygon_id=001

### preprocess_landsat_single_scene_10m.py

In [7]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from scipy.io import savemat
from pathlib import Path


# ============================================================
# Path setting
# ============================================================

BASE_DIR = Path("/mnt/disk1/workspace_jym/LCZ/data")

GT_50M_PATH = BASE_DIR / "GT" / "seoul_LCZ.tif"

RAW_SR_DIR = BASE_DIR / "Satellite" / "raw_SR"
RAW_ST_DIR = BASE_DIR / "Satellite" / "raw_ST"

OUT_10M_DIR = BASE_DIR / "Satellite" / "processed" / "10m"
OUT_NORM_DIR = BASE_DIR / "Satellite" / "processed" / "norm"

TARGET_RES = 10.0
NODATA_OUT = -9999.0


# ============================================================
# Utility functions
# ============================================================

def find_one(root: Path, pattern: str) -> Path:
    files = sorted(root.glob(pattern))
    if len(files) == 0:
        raise FileNotFoundError(f"No file found: {root} / {pattern}")
    if len(files) > 1:
        print(f"[WARN] Multiple files found for {pattern}. Use first one:")
        for f in files:
            print("   ", f)
    return files[0]


def make_qa_bad_mask(qa: np.ndarray) -> np.ndarray:
    """
    Landsat Collection 2 QA_PIXEL mask.

    Masked bits:
      bit 0: Fill
      bit 1: Dilated Cloud
      bit 2: Cirrus
      bit 3: Cloud
      bit 4: Cloud Shadow
      bit 5: Snow
    """
    qa = qa.astype(np.uint32)

    fill = (qa & (1 << 0)) > 0
    dilated_cloud = (qa & (1 << 1)) > 0
    cirrus = (qa & (1 << 2)) > 0
    cloud = (qa & (1 << 3)) > 0
    cloud_shadow = (qa & (1 << 4)) > 0
    snow = (qa & (1 << 5)) > 0

    return fill | dilated_cloud | cirrus | cloud | cloud_shadow | snow


def reproject_to_target(src_arr, src_profile, target_profile):
    src_arr = src_arr.astype(np.float32)
    src_arr = np.where(np.isfinite(src_arr), src_arr, NODATA_OUT).astype(np.float32)

    dst_arr = np.full(
        (target_profile["height"], target_profile["width"]),
        NODATA_OUT,
        dtype=np.float32
    )

    reproject(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        src_nodata=NODATA_OUT,
        dst_transform=target_profile["transform"],
        dst_crs=target_profile["crs"],
        dst_nodata=NODATA_OUT,
        resampling=Resampling.bilinear,
    )

    return dst_arr


def write_float_tif(out_path: Path, arr: np.ndarray, profile: dict):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    out_profile = profile.copy()
    out_profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=NODATA_OUT,
        compress="lzw"
    )

    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(arr.astype(np.float32), 1)


def normalize_and_save_mat(tif_paths, norm_dir: Path):
    norm_dir.mkdir(parents=True, exist_ok=True)

    for i, tif_path in enumerate(tif_paths, start=1):
        with rasterio.open(tif_path) as src:
            arr = src.read(1).astype(np.float32)
            nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        valid = np.isfinite(arr)
        if valid.sum() == 0:
            raise ValueError(f"No valid pixels in {tif_path}")

        vmin = np.nanmin(arr)
        vmax = np.nanmax(arr)

        if vmax == vmin:
            norm = np.zeros_like(arr, dtype=np.float32)
        else:
            norm = (arr - vmin) / (vmax - vmin)

        norm[~np.isfinite(norm)] = 0
        norm = norm.astype(np.float32)

        out_mat = norm_dir / f"b{i}_norm.mat"
        savemat(out_mat, {"norm": norm})

        print(
            f"Saved {out_mat.name} <- {tif_path.name} | "
            f"min={vmin:.4f}, max={vmax:.4f}"
        )


# ============================================================
# 1. Make target 10m grid from 50m LCZ GT
# ============================================================

with rasterio.open(GT_50M_PATH) as gt:
    gt_bounds = gt.bounds
    gt_crs = gt.crs

    target_width = gt.width * 5
    target_height = gt.height * 5

    target_transform = from_origin(
        gt_bounds.left,
        gt_bounds.top,
        TARGET_RES,
        TARGET_RES
    )

target_profile = {
    "driver": "GTiff",
    "height": target_height,
    "width": target_width,
    "count": 1,
    "dtype": "float32",
    "crs": gt_crs,
    "transform": target_transform,
    "nodata": NODATA_OUT,
}

print("========== Target 10m Grid ==========")
print("CRS:", gt_crs)
print("Width:", target_width)
print("Height:", target_height)
print("Transform:", target_transform)

# Seoul GT 기준이면 아래와 같아야 함
print("Expected shape: height=4245, width=4495")


# ============================================================
# 2. QA mask
# ============================================================

# SR 쪽 QA_PIXEL 우선 사용
try:
    qa_path = find_one(RAW_SR_DIR, "*QA_PIXEL*.TIF")
except FileNotFoundError:
    qa_path = find_one(RAW_ST_DIR, "*QA_PIXEL*.TIF")

with rasterio.open(qa_path) as qa_src:
    qa = qa_src.read(1)
    qa_bad = make_qa_bad_mask(qa)

print("\n========== QA ==========")
print("QA file:", qa_path)
print("Bad QA pixels:", int(qa_bad.sum()))


# ============================================================
# 3. Process SR_B1 ~ SR_B7
# ============================================================

OUT_10M_DIR.mkdir(parents=True, exist_ok=True)
output_tifs = []

print("\n========== Surface Reflectance ==========")

for b in range(1, 8):
    band_path = find_one(RAW_SR_DIR, f"*SR_B{b}.TIF")

    with rasterio.open(band_path) as src:
        dn = src.read(1).astype(np.float32)
        src_profile = src.profile.copy()

    # Landsat Collection 2 Level-2 Surface Reflectance scale factor
    arr = dn * 0.0000275 - 0.2

    # QA mask
    arr[qa_bad] = np.nan

    # reflectance range filtering
    arr[(arr < 0) | (arr > 1)] = np.nan

    arr_10m = reproject_to_target(arr, src_profile, target_profile)

    out_path = OUT_10M_DIR / f"{b:02d}_SR_B{b}_10m.tif"
    write_float_tif(out_path, arr_10m, target_profile)

    output_tifs.append(out_path)
    print(f"Saved: {out_path}")


# ============================================================
# 4. Process ST_B10
# ============================================================

print("\n========== Surface Temperature ==========")

st_path = find_one(RAW_ST_DIR, "*ST_B10.TIF")

with rasterio.open(st_path) as src:
    dn = src.read(1).astype(np.float32)
    src_profile = src.profile.copy()

# Landsat Collection 2 Level-2 Surface Temperature scale factor
lst_c = dn * 0.00341802 + 149.0 - 273.15

# QA mask
lst_c[qa_bad] = np.nan

# reasonable LST range filtering
lst_c[(lst_c < -50) | (lst_c > 80)] = np.nan

lst_10m = reproject_to_target(lst_c, src_profile, target_profile)

out_path = OUT_10M_DIR / "08_ST_B10_LST_C_10m.tif"
write_float_tif(out_path, lst_10m, target_profile)

output_tifs.append(out_path)
print(f"Saved: {out_path}")


# ============================================================
# 5. Normalization to .mat
# ============================================================

output_tifs = sorted(output_tifs)

print("\n========== Normalization order ==========")
for i, p in enumerate(output_tifs, start=1):
    print(f"b{i}_norm.mat <- {p.name}")

normalize_and_save_mat(output_tifs, OUT_NORM_DIR)

print("\n========== Done ==========")
print("10m GeoTIFFs saved to:", OUT_10M_DIR)
print("Normalized MAT files saved to:", OUT_NORM_DIR)

========== Target 10m Grid ==========
CRS: EPSG:32652
Width: 4495
Height: 4245
Transform: | 10.00, 0.00, 300630.00|
| 0.00,-10.00, 4180100.00|
| 0.00, 0.00, 1.00|
Expected shape: height=4245, width=4495

========== QA ==========
QA file: /mnt/disk1/workspace_jym/LCZ/data/Satellite/raw_SR/LC08_L2SP_116034_20200428_20200820_02_T1_QA_PIXEL.TIF
Bad QA pixels: 22467717

========== Surface Reflectance ==========
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/01_SR_B1_10m.tif
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/02_SR_B2_10m.tif
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/03_SR_B3_10m.tif
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/04_SR_B4_10m.tif
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/05_SR_B5_10m.tif
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/06_SR_B6_10m.tif
Saved: /mnt/disk1/workspace_jym/LCZ/data/Satellite/processed/10m/07_SR_B7_10m.tif

========== Surf

In [8]:
import rasterio
import numpy as np
from scipy.io import loadmat
from pathlib import Path

base_dir = Path("/mnt/disk1/workspace_jym/LCZ/data")

tif_dir = base_dir / "Satellite" / "processed" / "10m"
norm_dir = base_dir / "Satellite" / "processed" / "norm"
gt_path = base_dir / "GT" / "seoul_LCZ.tif"

print("========== GT 50m ==========")
with rasterio.open(gt_path) as src:
    print("GT shape:", src.height, src.width)
    print("GT CRS:", src.crs)
    print("GT res:", src.res)
    print("GT bounds:", src.bounds)

print("\n========== Processed 10m TIF ==========")
tif_paths = sorted(tif_dir.glob("*.tif"))

for p in tif_paths:
    with rasterio.open(p) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        valid = arr != nodata if nodata is not None else np.isfinite(arr)
        valid_ratio = valid.sum() / valid.size * 100

        print(p.name)
        print("  shape:", src.height, src.width)
        print("  crs:", src.crs)
        print("  res:", src.res)
        print("  bounds:", src.bounds)
        print("  nodata:", nodata)
        print("  valid ratio:", f"{valid_ratio:.2f}%")
        print("  min/max:", np.nanmin(arr[valid]), np.nanmax(arr[valid]))

print("\n========== Normalized MAT ==========")
mat_paths = sorted(norm_dir.glob("b*_norm.mat"))

for p in mat_paths:
    data = loadmat(p)
    norm = data["norm"]

    print(p.name)
    print("  shape:", norm.shape)
    print("  min:", np.nanmin(norm))
    print("  max:", np.nanmax(norm))
    print("  mean:", np.nanmean(norm))
    print("  zero ratio:", f"{(norm == 0).sum() / norm.size * 100:.2f}%")

========== GT 50m ==========
GT shape: 849 899
GT CRS: EPSG:32652
GT res: (50.0, 50.0)
GT bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)

========== Processed 10m TIF ==========
01_SR_B1_10m.tif
  shape: 4245 4495
  crs: EPSG:32652
  res: (10.0, 10.0)
  bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)
  nodata: -9999.0
  valid ratio: 99.79%
  min/max: 7.495284e-06 0.5948591
02_SR_B2_10m.tif
  shape: 4245 4495
  crs: EPSG:32652
  res: (10.0, 10.0)
  bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)
  nodata: -9999.0
  valid ratio: 99.82%
  min/max: 0.00050249696 0.7227811
03_SR_B3_10m.tif
  shape: 4245 4495
  crs: EPSG:32652
  res: (10.0, 10.0)
  bounds: BoundingBox(left=300630.0, bottom=4137650.0, right=345580.0, top=4180100.0)
  nodata: -9999.0
  valid ratio: 99.82%
  min/max: 0.0083249165 0.70927954
04_SR_B4_10m.tif
  shape: 4245 4495
  crs: EPSG:32652
  res: (10.0, 10.0)
  bound